In [ ]:
import pandas as pd
import pandas
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
import time 

warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
warnings.filterwarnings('ignore', category=Warning) 

print("A combinar XGBoost (v11) e LightGBM (v12) com um Meta-Modelo...")

def formatar_celula(series_coluna):
    s = series_coluna.astype(str).replace('NULL', pd.NA)
    s = s.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    s = s.str.upper()
    s = s.str.replace(r'[^A-Z0-9]+', '_', regex=True)
    s = s.str.strip('_')
    s = s.replace('', pd.NA)
    return s

def preprocessar_dados(df, colunas_scaler_treinadas=None, scaler=None):
    if 'RowId' not in df.columns and 'AVERAGE_SPEED_DIFF' not in df.columns:
        df_final_row_ids = np.arange(1, len(df) + 1)
    else:
        df_final_row_ids = None

    cols_to_transform = ['AVERAGE_CLOUDINESS', 'AVERAGE_RAIN']
    for col in cols_to_transform:
        if col in df.columns:
            df[col] = formatar_celula(df[col])

    cols_to_drop_base = ['city_name', 'AVERAGE_RAIN', 'AVERAGE_PRECIPITATION', 'record_date']
    
    try:
        df['record_date_dt'] = pd.to_datetime(df['record_date'], format='mixed', dayfirst=True)
        df['Hora_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.hour/24)
        df['Hora_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.hour/24)
        df['Mes_sin'] = np.sin(2 * np.pi * df['record_date_dt'].dt.month/12)
        df['Mes_cos'] = np.cos(2 * np.pi * df['record_date_dt'].dt.month/12)
        df['DIA_SEMANA'] = df['record_date_dt'].dt.dayofweek
        df['IS_WEEKEND'] = df['DIA_SEMANA'].isin([5, 6]).astype(int)
        rush_hours = [7, 8, 9, 17, 18, 19]
        df['IS_RUSH_HOUR'] = df['record_date_dt'].dt.hour.isin(rush_hours).astype(int)
        df = df.drop(columns=['record_date_dt'])
    except KeyError:
        pass

    cols_existentes_drop = [col for col in cols_to_drop_base if col in df.columns]
    df = df.drop(columns=cols_existentes_drop)

    if 'AVERAGE_CLOUDINESS' in df.columns:
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('NAN', np.nan)
        df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].fillna('NONE')

    cols_to_onehot = ['LUMINOSITY', 'AVERAGE_CLOUDINESS', 'DIA_SEMANA']
    for col in cols_to_onehot:
        if col in df.columns:
            prefix = col[:3].upper()
            if col == 'DIA_SEMANA': prefix = 'DAY'
            dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int, sparse=False)
            df = pd.concat([df, dummies], axis=1)
            df = df.drop(col, axis=1)

    cols_to_normalize = [
        'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME',
        'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
        'AVERAGE_WIND_SPEED', 'IS_WEEKEND', 'IS_RUSH_HOUR',
        'Hora_sin', 'Hora_cos', 'Mes_sin', 'Mes_cos'
    ]
    cols_existentes_normalize = [col for col in cols_to_normalize if col in df.columns]
    
    if scaler is None:
        scaler = MinMaxScaler()
        if cols_existentes_normalize:
            df[cols_existentes_normalize] = scaler.fit_transform(df[cols_existentes_normalize])
        return df, scaler, cols_existentes_normalize, df_final_row_ids
    else:
        cols_para_scaler_teste = [col for col in colunas_scaler_treinadas if col in df.columns]
        if cols_para_scaler_teste:
            df[cols_para_scaler_teste] = scaler.transform(df[cols_para_scaler_teste])
        return df, None, None, df_final_row_ids

print("\n--- [Passo 1/7] A carregar dados de Treino e Teste... ---")
df_train = pd.read_csv("training_data.csv", delimiter=",", encoding="latin-1")
df_test = pd.read_csv("test_data.csv", delimiter=",", encoding="latin-1")
y_train_raw = df_train.pop('AVERAGE_SPEED_DIFF')
test_row_ids = np.arange(1, len(df_test) + 1)

print("--- [Passo 2/7] A pré-processar dados de treino... ---")
X_train, scaler, colunas_scaler, _ = preprocessar_dados(df_train)
le = LabelEncoder()
y_train_formatado = formatar_celula(y_train_raw).replace('NAN', 'NONE').fillna('NONE')
y_train_encoded = le.fit_transform(y_train_formatado)


print("--- [Passo 3/7] A pré-processar dados de TESTE... ---")
X_test, _, _, _ = preprocessar_dados(df_test, colunas_scaler_treinadas=colunas_scaler, scaler=scaler)

print("--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---")
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train = X_train.reindex(columns=X_test.columns, fill_value=0)


print("\n--- [Passo 5/7] Hiperparâmetros dos base learners (modelo global) ---")
estimators = [
    ('xgb', XGBClassifier(
        learning_rate=0.08, max_depth=5, n_estimators=100,
        subsample=0.7, colsample_bytree=0.9,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=42, n_jobs=-1
    )),
    ('lgbm', LGBMClassifier(
        objective='multiclass',
        num_class=5,
        boosting_type='gbdt',
        learning_rate=0.04,
        num_leaves=31,
        n_estimators=300,
        min_data_in_leaf=40,
        feature_fraction=0.75,
        bagging_fraction=0.75,
        bagging_freq=3,
        lambda_l1=1.5,
        lambda_l2=1.5,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
]

meta_model = LogisticRegression(random_state=42, n_jobs=-1, max_iter=1000)

stack_clf = StackingClassifier(
    estimators=estimators, 
    final_estimator=meta_model, 
    cv=5,
    n_jobs=-1
)

print("A calcular o Cross-Validation do Stack (modelo global)")
start_time = time.time()
scores = cross_val_score(stack_clf, X_train, y_train_encoded, cv=5, scoring='accuracy', n_jobs=1, verbose=1)
end_time = time.time()
print(f"Accuracy CV (modelo global): {np.mean(scores):.5f}")

from sklearn.metrics import f1_score, accuracy_score

print("\n--- [Passo 6/7] Treinar o modelo multi classe base ---")
stack_clf.fit(X_train, y_train_encoded)

# ==================================================
#       SISTEMA DE ESPECIALISTAS — ULTRA
# ==================================================

classes = le.classes_  # ['HIGH','LOW','MEDIUM','NONE','VERY_HIGH']
especialistas = {}
scores_f1 = {}

print("\n=== [ULTRA] A treinar especialistas One-vs-All ===")

for classe in classes:
    print(f"\n🔍 Treinar especialista para: {classe}")

    y_bin = (y_train_formatado == classe).astype(int)

    pos = int((y_bin == 1).sum())
    neg = int((y_bin == 0).sum())

    weight = (neg/pos) if pos > 0 else 1
    weight = min(max(weight, 1.0), 10.0)  # peso controlado

    especialista = StackingClassifier(
        estimators=[
            ("xgb", XGBClassifier(
                learning_rate=0.05,
                max_depth=4,
                n_estimators=120,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric='logloss',
                random_state=42,
                scale_pos_weight=weight
            )),
            ("lgbm", LGBMClassifier(
                objective="binary",
                n_estimators=160,
                num_leaves=24,
                min_data_in_leaf=35,
                learning_rate=0.03,
                feature_fraction=0.8,
                bagging_fraction=0.8,
                bagging_freq=3,
                lambda_l1=1.0,
                lambda_l2=1.0,
                class_weight={0: 1.0, 1: weight},
                random_state=42
            ))
        ],
        final_estimator=LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        ),
        cv=3,
        n_jobs=-1
    )

    especialista.fit(X_train, y_bin)
    especialistas[classe] = especialista

    preds_local = especialista.predict(X_train)
    f1 = f1_score(y_bin, preds_local)
    scores_f1[classe] = max(f1, 0.3)
    print(f"   → F1 Score especialista {classe}: {f1:.4f}")

# ==================================================
#  FUSÃO ULTRA (GLOBAL + ESPECIALISTAS)
# ==================================================

proba_global = stack_clf.predict_proba(X_test)
df_global = pd.DataFrame(proba_global, columns=classes)

df_spec_raw = pd.DataFrame()
for classe, modelo in especialistas.items():
    p = modelo.predict_proba(X_test)[:,1]
    df_spec_raw[classe] = p * scores_f1[classe]

row_sums = df_spec_raw.sum(axis=1).replace(0,np.nan)
df_spec_norm = df_spec_raw.div(row_sums, axis=0).fillna(0.0)

alpha = 0.30  # peso dos especialistas

df_blend = (1-alpha)*df_global + alpha*df_spec_norm

if "VERY_HIGH" in df_blend.columns:
    boost = df_spec_raw["VERY_HIGH"]>0.6
    df_blend.loc[boost,"VERY_HIGH"]+=0.05

df_blend = df_blend.div(df_blend.sum(axis=1),axis=0)

pred_final = df_blend.idxmax(axis=1).str.title()

submission_df = pd.DataFrame({
    "RowId":test_row_ids,
    "Speed_Diff":pred_final
})
submission_df.to_csv("submission_especialistas_ULTRA.csv",index=False)

print("\n============= ULTRA FINALIZADO =============")
print(" Submission gerada -> submission_especialistas_ULTRA.csv")
print("============================================")

# METRICA REAL EM TREINO

pred_train_global = stack_clf.predict(X_train)
acc_g = accuracy_score(y_train_encoded,pred_train_global)

proba_global_train = stack_clf.predict_proba(X_train)
df_global_train = pd.DataFrame(proba_global_train,columns=classes)

df_spec_train = pd.DataFrame()
for classe,modelo in especialistas.items():
    p = modelo.predict_proba(X_train)[:,1]
    df_spec_train[classe] = p*scores_f1[classe]

row_sums_t = df_spec_train.sum(axis=1).replace(0,np.nan)
df_spec_train_norm = df_spec_train.div(row_sums_t,axis=0).fillna(0.0)

df_blend_train = (1-alpha)*df_global_train + alpha*df_spec_train_norm
df_blend_train = df_blend_train.div(df_blend_train.sum(axis=1),axis=0)

pred_train_ultra = df_blend_train.idxmax(axis=1)

acc_ultra = accuracy_score(y_train_formatado,pred_train_ultra)

print(f"\n🔵 Accuracy Global (stack): {acc_g:.4f}")
print(f"🟣 Accuracy ULTRA (train): {acc_ultra:.4f}  <== este é o que interessa\n")



A combinar XGBoost (v11) e LightGBM (v12) com um Meta-Modelo...

--- [Passo 1/7] A carregar dados de Treino e Teste... ---
--- [Passo 2/7] A pré-processar dados de treino... ---
--- [Passo 3/7] A pré-processar dados de TESTE... ---
--- [Passo 4/7] A alinhar colunas de Treino e Teste... ---

--- [Passo 5/7] Hiperparâmetros dos base learners (modelo global) ---
A calcular o Cross-Validation do Stack (modelo global)


[17:34:50] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:34:50] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:34:50] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:34:50] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:34:50] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:35:25] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:35:25] WARNING: /Users/runner/minifo

Accuracy CV (modelo global): 0.81048

--- [Passo 6/7] Treinar o modelo multi classe base ---


[17:37:39] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:37:39] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:37:39] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:37:39] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[17:37:39] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1758007651359/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

